In [1]:
import pandas as pd
import numpy as np

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding, TimeDistributed
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

In [2]:
df = pd.read_csv("NER dataset.csv", encoding="latin1")

df = df.fillna(method="ffill")  # fill missing sentence numbers

print(df.head())

/tmp/ipykernel_5915/3493586331.py:3: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method="ffill")  # fill missing sentence numbers


    Sentence #           Word  POS Tag
0  Sentence: 1      Thousands  NNS   O
1  Sentence: 1             of   IN   O
2  Sentence: 1  demonstrators  NNS   O
3  Sentence: 1           have  VBP   O
4  Sentence: 1        marched  VBN   O


In [3]:
sentences = []

grouped = df.groupby("Sentence #")

for _, group in grouped:
    words = list(group["Word"].values)
    tags = list(group["Tag"].values)
    sentences.append((words, tags))

print(sentences[0])

(['Thousands', 'of', 'demonstrators', 'have', 'marched', 'through', 'London', 'to', 'protest', 'the', 'war', 'in', 'Iraq', 'and', 'demand', 'the', 'withdrawal', 'of', 'British', 'troops', 'from', 'that', 'country', '.'], ['O', 'O', 'O', 'O', 'O', 'O', 'B-geo', 'O', 'O', 'O', 'O', 'O', 'B-geo', 'O', 'O', 'O', 'O', 'O', 'B-gpe', 'O', 'O', 'O', 'O', 'O'])


In [4]:
words = list(set(df["Word"].values))
words.append("PAD")

tags = list(set(df["Tag"].values))

word2idx = {w: i+1 for i, w in enumerate(words)}
tag2idx = {t: i for i, t in enumerate(tags)}

idx2tag = {i: t for t, i in tag2idx.items()}

In [5]:
X = [[word2idx[w] for w in sent[0]] for sent in sentences]
y = [[tag2idx[t] for t in sent[1]] for sent in sentences]

In [6]:
max_len = 50

X = pad_sequences(X, maxlen=max_len, padding="post", value=word2idx["PAD"])
y = pad_sequences(y, maxlen=max_len, padding="post", value=tag2idx["O"])

In [7]:
y = [to_categorical(i, num_classes=len(tags)) for i in y]

In [8]:
model = Sequential()

model.add(Embedding(input_dim=len(words)+1, output_dim=50, input_length=max_len))
model.add(LSTM(50, return_sequences=True))
model.add(TimeDistributed(Dense(len(tags), activation="softmax")))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [9]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ ?                      │   0 (unbuilt) │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [10]:
model.fit(
    X,
    np.array(y),
    batch_size=32,
    epochs=3,
    verbose=1
)

Epoch 1/3
1499/1499 ━━━━━━━━━━━━━━━━━━━━ 75s 47ms/step - accuracy: 0.9527 - loss: 0.2108
Epoch 2/3
1499/1499 ━━━━━━━━━━━━━━━━━━━━ 73s 48ms/step - accuracy: 0.9829 - loss: 0.0630
Epoch 3/3
1499/1499 ━━━━━━━━━━━━━━━━━━━━ 80s 47ms/step - accuracy: 0.9862 - loss: 0.0454


In [11]:
test_sentence = ["EU", "rejects", "German", "call"]

test_seq = [word2idx.get(w, 0) for w in test_sentence]
test_seq = pad_sequences([test_seq], maxlen=max_len, padding="post")

pred = model.predict(test_seq)

pred_tags = [idx2tag[np.argmax(p)] for p in pred[0][:len(test_sentence)]]

print("Sentence:", test_sentence)
print("Predicted Tags:", pred_tags)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 618ms/step
Sentence: ['EU', 'rejects', 'German', 'call']
Predicted Tags: ['B-org', 'O', 'B-gpe', 'O']
